# 04 · Detection report

Out of N fake videos, how many are caught. Out of N real videos, how many are
cleared. Plus the false positives and false negatives behind those counts, and
what every threshold would have cost.

**Samples an equal number of each class.** The test split is ~76% fake, so raw
accuracy on it flatters any model that leans toward "fake". Fixing the counts at
N vs N makes the two error types directly comparable.

Run the setup cell, then whichever report you want — they're independent.

## 1 · Setup

In [ ]:
import os, torch, subprocess

print(subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "no GPU")
print("cuda:", torch.cuda.is_available(), " (CPU works too, just slower)")

!pip install -q timm scikit-learn pandas tqdm

REPO = "/content/deepfake_system"
if not os.path.exists(REPO):
    !git clone -q https://github.com/sailessawesome-ui/deepfake_system.git {REPO}
else:
    !cd {REPO} && git pull -q
WORK = f"{REPO}/deepfake_system"
RUNS = "/content/drive/MyDrive/deepfake_runs"
!cd {WORK} && git log --oneline -1

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2 · What have you got

The report needs the crops on local disk. If the manifests or `/content/data`
are missing, this session is fresh — go back to notebook 03 and re-run its
unzip and manifest cells first.

In [ ]:
import os

MODELS = {
    "A_full":           "3 sources, wild held out - the reported science",
    "B_no_consistency": "same data, consistency loss off - ablation",
    "C_mesonet":        "MesoInception-4, the cited architecture",
    "D_deployed":       "all 4 sources incl. wild - the shipped model",
}

# A, B and C were trained without wild, so they must be scored on the
# manifest that matches. D trained on the split that includes it.
MANIFEST_ABC = "/content/manifest_runA.csv"
MANIFEST_D   = "/content/manifest.csv"

print("CHECKPOINTS")
have = []
for name, desc in MODELS.items():
    p = f"{RUNS}/{name}/best.pt"
    ok = os.path.exists(p)
    if ok:
        have.append(name)
    print(f"  {'yes' if ok else ' - ':4s} {name:18s} {desc}")

print()
print("MANIFESTS")
for m in (MANIFEST_ABC, MANIFEST_D):
    print(f"  {'yes' if os.path.exists(m) else ' - ':4s} {m}")

print()
print("DATA")
for d in ("FF++", "celebdf_faces", "df40_frames", "wild"):
    p = f"/content/data/{d}"
    print(f"  {'yes' if os.path.isdir(p) and os.listdir(p) else ' - ':4s} {d}")

## 3 · One report

Change `MODEL` and `SPLIT`, run it. Add `--degraded` to score
messenger-transcoded copies instead of clean ones.

`--threshold 0.45` overrides `calibration.json` if you want to see what a
different cut point would do without re-running evaluation.

In [ ]:
MODEL = "A_full"        # A_full | B_no_consistency | C_mesonet | D_deployed
SPLIT = "test"          # test | holdout | unseen_method
N     = 200             # videos per class

manifest = MANIFEST_D if MODEL == "D_deployed" else MANIFEST_ABC
print(f"{MODEL} on '{SPLIT}', {N} fake + {N} real, manifest {manifest}\n")

In [ ]:
!cd {WORK} && python -m scripts.detection_report \
    --checkpoint {RUNS}/{MODEL}/best.pt \
    --manifest {manifest} --split {SPLIT} --n {N}

## 4 · Every model, every condition

The full sweep. Takes a while — each report scores 2N videos with TTA — so run
it once and read the summary table afterwards.

Conditions: **clean** and **degraded** on `test`, plus `holdout` for the models
that have one (D trained on wild, so it has no holdout split by design).

In [ ]:
import itertools, os

JOBS = []
for name in have:
    mani = MANIFEST_D if name == "D_deployed" else MANIFEST_ABC
    if not os.path.exists(mani):
        continue
    JOBS.append((name, mani, "test", False))
    JOBS.append((name, mani, "test", True))
    if name != "D_deployed":
        JOBS.append((name, mani, "holdout", False))

print(f"{len(JOBS)} reports queued:")
for n, _, s, d in JOBS:
    print(f"   {n:18s} {s:14s} {'degraded' if d else 'clean'}")

In [ ]:
for name, mani, split, degraded in JOBS:
    flag = "--degraded" if degraded else ""
    print("\n" + "#" * 68)
    print(f"# {name}  |  {split}  |  {'degraded' if degraded else 'clean'}")
    print("#" * 68, flush=True)
    !cd {WORK} && python -m scripts.detection_report --checkpoint {RUNS}/{name}/best.pt --manifest {mani} --split {split} --n {N} {flag}

## 5 · Summary table

Reads the JSON each report wrote and assembles one table — paste-ready markdown
for your write-up.

In [ ]:
import json, os
import pandas as pd

recs = []
for name in MODELS:
    for split in ("test", "holdout", "unseen_method"):
        for cond in ("clean", "degraded"):
            p = f"{RUNS}/{name}/detection_{split}_{cond}.json"
            if not os.path.exists(p):
                continue
            d = json.load(open(p))
            c, r = d["confusion"], d["rates"]
            recs.append({
                "model": name, "split": split, "condition": cond,
                "fake_caught": f"{c['tp']}/{d['n_fake']}",
                "real_cleared": f"{c['tn']}/{d['n_real']}",
                "FP": c["fp"], "FN": c["fn"],
                "detection": round(r["recall"], 3),
                "specificity": round(r["specificity"], 3),
                "bal_acc": round(r["balanced_acc"], 3),
                "AUC": round(r["auc"], 3) if "auc" in r else None,
                "thr": round(d["threshold"], 3),
            })

if not recs:
    print("No detection_*.json found yet - run section 3 or 4 first.")
else:
    df = pd.DataFrame(recs).sort_values(["model", "split", "condition"])
    print(df.to_string(index=False))
    print()
    print("MARKDOWN")
    print(df.to_markdown(index=False))

---

## Reading it

**`detection`** is the fraction of fakes caught — recall. **`specificity`** is
the fraction of genuine videos correctly cleared. They trade against each other:
lowering the threshold raises one and lowers the other.

**`bal_acc`** is the average of the two, and it's the honest single number when
the classes are balanced by construction the way they are here.

**`AUC`** is threshold-independent — it's the model's actual ability to
separate the classes. No threshold change improves it. If AUC is 0.78, no cut
point gets you a reliable detector; only different training data does.

### What to look for

**A vs B on `test / degraded`** — this is your novelty claim. If A clears more
real videos and catches more fakes than B under transcoding, the consistency
loss earned its place. Compare the `bal_acc` column.

**A vs D on their respective splits** — D has WildDeepfake in training, so it
should handle in-the-wild footage far better. Not a like-for-like comparison,
and shouldn't be presented as one.

**`holdout` rows** — the honest generalisation number. Expect it to be the
worst row in the table. Cross-dataset AUC of 0.65–0.78 is the published norm,
so a low number here is a correct measurement, not a broken model.

### If specificity is very low

The threshold is too aggressive. Look at the sweep each report prints and find
the balanced-accuracy optimum, then re-run `evaluate.py` with
`--threshold-criterion youden` so `calibration.json` picks it up and the app
inherits it.